(rd.comparing)=
# 13. Comparing Lattices

In [1]:
import numpy as np
import scibmad as sb
sb.options.display_width = 100
sb.options.display_height = 30

Bmad's program Tao keeps three copies of a lattice: the "design" lattice as read from the lattice
file, which never changes, the "model" lattice, which the commands of Tao change, and a "base"
lattice, a copy of the model made at some point to compare with. In SciBmad, a lattice is a set of
Julia objects, which you change directly. To compare the lattice before and after a change, keep
the result of `twiss` from before the change: it does not change when the lattice does.

## Example

An open line with a drift, a bend and a quadrupole, for electrons with a total energy of 10 MeV,
starting with beta functions of 10 m:

In [2]:
import numpy as np
import scibmad as sb

with sb.elements():
    d = sb.Drift(L=0.5)
    b = sb.SBend(L=0.5, g_ref=1.0, Kn0=1.001, e1=0.1)
    q = sb.Quadrupole(L=0.6, Kn1=0.23)
for ele in (d, b, q):
    ele.n_steps = 10  # For smooth plots
simple = sb.Beamline([d, b, q], species_ref=sb.Species("electron"), E_ref=10e6)
initial = dict(beta1=10.0, beta2=10.0)
tw_design = sb.twiss(simple, initial=initial)
tw_design

Twiss:
summ:
 q1     = 0.03131247815390913
 q2     = 0.027056500923310512
 etac   = 0.010153945554962213
 alphac = 0.012765144831025022

df:
31×26 DataFrame
 Row │ index  name  kind        s     beta1     phi1         dx        x             beta2     phi ⋯
     │                          [m]   [m]       [2π]         [m]       [m]           [m]       [2π ⋯
─────┼──────────────────────────────────────────────────────────────────────────────────────────────
   1 │     1  d     Drift       0.0   10.0      0.0          0.0        0.0          10.0      0.0 ⋯
   2 │     1  d     Drift       0.05  10.0003   0.000795768  0.0        0.0          10.0003   0.0
   3 │     1  d     Drift       0.1   10.001    0.0015915    0.0        0.0          10.001    0.0
   4 │     1  d     Drift       0.15  10.0023   0.00238715   0.0        0.0          10.0023   0.0
   5 │     1  d     Drift       0.2   10.004    0.00318267   0.0        0.0          10.004    0.0 ⋯
   6 │     1  d     Drift       0.25  10.

The field of the bend `Kn0` is a little stronger than the curvature `g_ref` of the reference
trajectory, so the orbit is not zero. Now the lattice is changed: a vertical kick of −0.5 mrad is
added to the drift (a skew dipole field, `Ks0L`), and a horizontal kick of 1 mrad to the quadrupole
(a normal dipole field, `Kn0L`: a positive normal dipole field kicks towards $-x$):

In [3]:
d.Ks0L = -0.0005
q.Kn0L = -0.001
tw = sb.twiss(simple, initial=initial)
tw

Twiss:
summ:
 q1     = 0.03131249665753308
 q2     = 0.027056514844851155
 etac   = 0.01023914891812972
 alphac = 0.012850348194192528

df:
31×26 DataFrame
 Row │ index  name  kind        s     beta1     phi1         dx        x             beta2     phi ⋯
     │                          [m]   [m]       [2π]         [m]       [m]           [m]       [2π ⋯
─────┼──────────────────────────────────────────────────────────────────────────────────────────────
   1 │     1  d     Drift       0.0   10.0      0.0          0.0        0.0          10.0      0.0 ⋯
   2 │     1  d     Drift       0.05  10.0003   0.000795768  0.0        0.0          10.0003   0.0
   3 │     1  d     Drift       0.1   10.001    0.0015915    0.0        0.0          10.001    0.0
   4 │     1  d     Drift       0.15  10.0023   0.00238715   0.0        0.0          10.0023   0.0
   5 │     1  d     Drift       0.2   10.004    0.00318267   0.0        0.0          10.004    0.0 ⋯
   6 │     1  d     Drift       0.25  10.0

The orbit of the changed lattice, and its difference from the design:

In [4]:
from matplotlib.figure import Figure
fig = Figure(figsize=(7, 5), layout="constrained")
ax1, ax2 = fig.subplots(2, 1)
ax1.plot(tw.s, 1e3*tw.x, label="x")
ax1.plot(tw.s, 1e3*tw.y, label="y")
ax1.set_ylabel("orbit [mm]")
ax1.legend(loc="upper left")
ax2.plot(tw.s, 1e3*(tw.x - tw_design.x), label="x")
ax2.plot(tw.s, 1e3*(tw.y - tw_design.y), label="y")
ax2.set(xlabel="s [m]", ylabel="orbit - design [mm]")
fig

<Figure size 700x500 with 2 Axes>

To compare with another state than the design, keep another `twiss` result, as Tao's "base"
lattice:

In [5]:
tw_base = tw
q.Ks0L = 0.0005
tw = sb.twiss(simple, initial=initial)
np.max(np.abs(tw.y - tw_base.y)), np.max(np.abs(tw.x - tw_base.x))

(np.float64(0.00015103793942826144), np.float64(1.2708047658074265e-10))

To set the lattice back, set the parameters back (`d.Ks0L = 0` etc.). A match can be undone with
{jl}`undo!`{py}`sb.undo` of its result, e.g. {jl}`res = match!(...)`{py}`res = sb.match(...)`, then
{jl}`undo!(res)`{py}`sb.undo(res)`.

## Exercises

1. **Orbit plots:** Plot the horizontal orbit of the design lattice, and that of the changed
   lattice, in one plot.

2. **Functions:** To save typing, define a function `setit(k)` that sets the vertical kick of `q` to
   `k`, so that `setit(1e-6)` is the same as `q.Ks0L = 1e-6`.

3. **Changing the tunes with the arcs:** Start with the ring of [chapter 9](rf.md), and keep its
   `twiss` result as the design. Then change the tunes to 54.10 and 54.12 by varying the arc
   quadrupoles `qf` and `qd` instead of the tune cell. Plot the relative difference of the beta functions from the design,
   `tw.beta1 ./ tw_design.beta1 .- 1`, around the ring. How much "beta beating" is there? Why does
   the tune cell avoid it?